In [ ]:
from torch.nn import *
import numpy as np
import torch

torch.set_default_tensor_type(torch.cuda.FloatTensor)


def unpack(a, n, m=2 ** torch.arange(16)):
    return (a[..., None] & m[:n]).bool().float()


def dat(k, c, len=0, bs=4, std=256, code="bin"):
    len = 2 ** int(np.log2((len or k + 1) * c))
    y = torch.randint(c, (bs, len))
    x = torch.randn(bs, len, k).add_(torch.randn(bs, c, k).mul_(std)[torch.arange(bs)[:, None], y])
    if code == "onehot":
        return torch.cat((x, functional.one_hot(y, c).float()), -1), x, y
    y = unpack(y, np.log2(c).__ceil__())
    return torch.cat((x, y), -1), x, y


def los(x, y, code="bin"):
    if code == "onehot":
        return functional.cross_entropy(x.transpose(1, 2), y)
    return functional.binary_cross_entropy_with_logits(x, y)


def acc(x, y, code="bin"):
    if code == "onehot":
        x = x.argmax(-1) == y
    else:
        x = (x.signbit() != y).all(-1)
    return x.sum() / x.numel()


def eval(m, d, x, y):
    return m[1:](d + m[0](x)), y


def evals(m, d, x, y):
    return m(torch.cat((d, x.broadcast_to(len(d), *x.shape)), 3)), y


class BatchNorm(BatchNorm1d):
    def forward(s, x):
        return super().forward(x.view(-1, x.size(-1))).view(x.shape)


def Nn(i, o=0, m=0, b=1, bn=0, act=Hardswish(1)):
    o = o or i
    m = m or max(i, o)
    return Sequential(Linear(i, m, not bn), *(BatchNorm1d(m, track_running_stats=0),) if bn else (), act, Linear(m, o, b)) if act else Linear(i, o, b)


class Tnn(Sequential):
    def __init__(s, *_, f=0):
        super().__init__(*_)
        s.f = f

    def forward(s, x):
        for _ in range(int(np.log2(x.size(1)))):
            x = super().forward(s.f(x.view(-1, 2, x.size(-1)), 1) if s.f else x.view(-1, 2 * x.size(-1)))
        return x


class Tnns(Tnn):
    def forward(s, x):
        n = len(x)
        x = x.view(-1, x.size(2))
        y = [x]
        while len(x) > n:
            x = super().forward(s.f(x.view(-1, 2, x.size(1)), 1) if s.f else x.view(-1, 2 * x.size(1)))
            y.append(x.repeat_interleave(y[0].numel() // x.numel(), 0))
        return torch.stack(y).view(len(y), n, -1, x.size(1))


class SelfAttention(Sequential):
    def forward(s, x):
        t = s[2](x).sigmoid_()
        return (x * t).add_(s[1](x * s[0](x).sigmoid_()) * (1 - t))


class SelfAttention2(Sequential):
    def forward(s, x):
        x = torch.cat((x, s[1](x * s[0](x).sigmoid_())), 1)
        return (x.view(len(x), 3, -1) * s[2](x).view(len(x), 3, -1).softmax(1)).sum(1)


class SelfAttentionLinear(Module):
    def __init__(s, k, L, K=0):
        super().__init__()
        s.r = torch.arange(float(L))[:, None]
        s.a = Linear(k, 1)
        s.b = Linear(k, K or k)

    def forward(s, x):
        return (s.r * s.a(x)[:, None, :, 0]).softmax(1) @ s.b(x)


class SelfAttentionNn(SelfAttentionLinear):
    def __init__(s, k, L, K=0, h=0):
        super().__init__(k, L, h)
        s.c = Linear(h or k, K or k)

    def forward(s, x):
        return s.c(functional.hardswish(super().forward(x), 1))


class SelfAttentionFullNn(Module):
    def __init__(s, k, L, K=0, h=0):
        super().__init__()
        s.r = torch.arange(float(L))[:, None]
        h = h or k
        s.a = Linear(1, h)
        s.b = Linear(k, h)
        s.c = Linear(h, h)
        s.d = Linear(k, h)
        s.e = Linear(h, K or k)

    def forward(s, x):
        return s.e(functional.hardswish((functional.hardswish(s.a(s.r), 1) @ s.c(functional.hardswish(s.b(x), 1)).transpose(1, 2)).softmax(1) @ s.d(x), 1))


def model(k, c, fac=1, exp=1, hfac=0, hexp=1):
    K = int(fac * (k * c) ** exp)
    h = max(int(hfac * (k * c) ** hexp), int((k * K) ** 0.5))
    M = Sequential(SelfAttentionNn(k + np.log2(c).__ceil__(), c, k), type("", (Module,), {"forward": lambda _, x: x.view(len(x), -1)})())
    m = Nn(k * c + k, np.log2(c).__ceil__())
    # M = Sequential(Linear(k + c, K), Hardswish(1), type("", (Module,), {"forward": lambda _, x: x.mean(1, True)})(), Linear(K, K, 0))
    # m = Sequential(Linear(k, K), Hardswish(1), Linear(K, c))
    # M = Sequential(Nn(k + c, K, h), Tnn(SelfAttention(Linear(K, K, 0), Sequential(Linear(K, K, 0), Hardswish(1)), Linear(K, K, 0)), torch.sum))
    # M = Sequential(Nn(k + c, K, h), Tnn(SelfAttention2(Linear(K * 2, K * 2, 0), Sequential(Linear(K * 2, K, 0), Hardswish(1)), Linear(K * 3, K * 3, 0))))
    # m = Nn(K + k, c, h)
    return M, m

In [ ]:
from torch.nn import *
import torch

torch.set_default_tensor_type(torch.cuda.FloatTensor)

def dat(k, c, bs=64, =1):
    y = torch.randint(c, (bs, len*c))
    x = torch.randn(bs, len, k).add_(torch.randn(bs, c, k).mul_(std)[torch.arange(bs)[:, None], y])

class Attention(Module):
    def __init__(s, Q=Identity(), K=Identity(), V=Identity(), q=None, k=None, v=None):
        super().__init__()
        s.Q = Q
        s.K = K
        s.V = V
        s.q = q
        s.k = k
        s.v = v

    def forward(s, q=None, k=None, v=None):
        if k is None:
            k = q
        if v is None:
            v = k
        return (s.Q(s.q if q is None else q) @ s.K(s.k if k is None else k).transpose(-2, -1)).softmax(-1) @ s.V(s.v if v is None else v)

In [ ]:
k = 2
c = 4
M = Attention(Identity(), Linear(1, 1), Linear(k, k), torch.arange(float(c))[:, None])
m = Attention(Linear(k, k), Linear(k, k), Identity(), v=Parameter(torch.eye(c)))

In [ ]:
M

In [ ]:
m

In [ ]:
x = torch.randn(100, k)
y = torch.randint(0, c, [100])

In [ ]:
m(q=x, k=M(k=y.float()[:,None], v=x)).softmax(1)

In [ ]:
torch.eye(4)

In [ ]:
Attention(Linear(1, 4), Linear(4, 4), Linear(4, 6), torch.arange(5.)[:,None])(torch.rand(2, 3, 4))

In [ ]:
attention(torch.rand(10,5,2),torch.rand(10,4,2),torch.rand(10,4,3)).shape

In [ ]:
k = 2
c = 4
M, m = model(k, c, fac=8)
o = torch.optim.Adam((*M.parameters(), *m.parameters()))
O = torch.optim.lr_scheduler.OneCycleLR(o, 0.01, 1000000, pct_start=0.01, div_factor=9, final_div_factor=9999)
s = torch.cuda.amp.GradScaler()
print(M, m, sum(i.numel() for i in o.param_groups[0]["params"]))
for i in range(O.total_steps // 10000):
    l = []
    a = []
    for i in range(10000):
        o.zero_grad()
        with torch.cuda.amp.autocast():
            t = dat(k, c, 1, 128)
            t = eval(m, M(t[0]), *t[1:])
            if not i % 1000:
                a.append(acc(*t))
            t = los(*t)
        s.scale(t).backward()
        s.step(o)
        s.update()
        O.step()
        l.append(t)
    print(sum(l) / len(l), sum(a) / len(a), o.param_groups[0]["lr"])